# LambdaG Visualization — De Machabeis vs. Comparison Candidates

This notebook generates the local token-level LambdaG visualizations for the hold-out work `X3_machabeis`.

The poem remains excluded from training and calibration. For each candidate in the shared hexameter test corpus, the visualization compares the complete masked *De Machabeis* with up to 2000 tokens of the candidate and 20,000 tokens from the reference population.

`X4_certamina` is also reintroduced here as an **analysis-only candidate**, exactly as in the comparative authorship-verification notebook. It remains excluded from training and calibration and is never used in the reference population for the other candidates.

The HTML files are saved locally in the poem-specific folder.


## Stage 1: Setup

The shared corpora and processed data are stored one directory above this notebook. HTML outputs remain in the current *De Machabeis* folder.


In [3]:
from pathlib import Path
import numpy as np

from sentence_corpora import TwoLevelCorpus
from lambdag.visualize import lambdaG_visualize

# ---------------------------------------------------------
# Shared files produced by the general hexameter pipeline
# ---------------------------------------------------------
GENERAL_DIR = Path("..").resolve()

TEST_CORPUS_PATH = GENERAL_DIR / "test_corpus.pkl"
MACHABEIS_CORPUS_PATH = GENERAL_DIR / "machabeis_corpus.pkl"
CERTAMINA_CORPUS_PATH = GENERAL_DIR / "certamina_corpus.pkl"
DATA_PROCESSED_DIR = GENERAL_DIR / "data_processed"

# Local output directory
OUTPUT_DIR = Path("html_machabeis")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for path in [
    TEST_CORPUS_PATH,
    MACHABEIS_CORPUS_PATH,
    CERTAMINA_CORPUS_PATH,
    DATA_PROCESSED_DIR,
]:
    if not path.exists():
        raise FileNotFoundError(f"Required path not found: {path}")

test_corpus = TwoLevelCorpus.from_pickle(str(TEST_CORPUS_PATH))
holdout_corpus = TwoLevelCorpus.from_pickle(str(MACHABEIS_CORPUS_PATH))
certamina_corpus = TwoLevelCorpus.from_pickle(str(CERTAMINA_CORPUS_PATH))

# Sanity checks: both uncertain texts must remain outside the shared test corpus
test_authors = set(test_corpus.get_unique_values("author"))
assert "X3" not in test_authors, "X3_machabeis must not be present in test_corpus"
assert "X4" not in test_authors, "X4_certamina must not be present in test_corpus"

assert set(holdout_corpus.get_unique_values("author")) == {"X3"}
assert set(certamina_corpus.get_unique_values("author")) == {"X4"}

print(f"General analysis directory: {GENERAL_DIR}")
print(f"HTML output directory: {OUTPUT_DIR.resolve()}")
print(f"test_corpus: {len(test_corpus)} sentences")
print(
    f"De Machabeis: {len(holdout_corpus)} sentences, "
    f"{sum(len(s.text.split()) for s in holdout_corpus)} tokens"
)
print(
    f"Certamina: {len(certamina_corpus)} sentences, "
    f"{sum(len(s.text.split()) for s in certamina_corpus)} tokens"
)
print(f"Authors in test_corpus: {sorted(test_corpus.get_unique_values('author'))}")


General analysis directory: /Users/martinapaccara/Documents/Documenti/Università/POST-LAUREA/DOTTORATO/Lavoro/Digitale/Stilometria/hexameters/Lexical analysis
HTML output directory: /Users/martinapaccara/Documents/Documenti/Università/POST-LAUREA/DOTTORATO/Lavoro/Digitale/Stilometria/hexameters/Lexical analysis/6. De Machabaeis/html_machabeis
test_corpus: 14066 sentences
De Machabeis: 213 sentences, 3623 tokens
Certamina: 55 sentences, 1153 tokens
Authors in test_corpus: ['AI', 'AeC', 'AeP', 'B', 'BC', 'DC', 'E', 'FB', 'GAp', 'GAr', 'GC', 'GM', 'GP', 'GVS', 'GiP', 'H', 'HA', 'HP', 'IH', 'II', 'M', 'NLC', 'OM', 'OSE', 'RC', 'RT', 'SG', 'U1', 'U10', 'U11', 'U2', 'U3', 'U4', 'U5', 'U6', 'U7', 'U8', 'U9']


## Stage 2: Visualization per Candidate

For each comparison candidate:

1. use up to 2000 tokens of candidate text;
2. use 20,000 reference tokens;
3. use the complete masked *De Machabeis* as questioned text;
4. align the masked sentences with the unmasked `.quesplit.txt` version when available;
5. save masked and unmasked HTML visualizations in `html_machabeis/`.

For the ordinary background candidates, the reference population consists of the other authors in `test_corpus`. For `X4` (*Certamina septem fratrum Machabeorum*), the candidate text comes from its hold-out corpus while the reference population remains the known-authorship `test_corpus`.


In [5]:
# Background candidates plus X4_certamina as analysis-only candidate
background_candidates = sorted(test_corpus.get_unique_values("author"))
candidates = background_candidates + ["X4"]

print(
    f"Found {len(background_candidates)} background candidates + "
    "X4 (Certamina)"
)

rng = np.random.default_rng(seed=42)

# Complete questioned text: De Machabeis
q_sentences = [s.text.split() for s in holdout_corpus]
print(f"Questioned sentences (X3_machabeis): {len(q_sentences)}")

# Load unmasked De Machabeis sentences once
unmasked_sents = None
quesplit_files = list(
    DATA_PROCESSED_DIR.glob("*machabeis_sentences.quesplit.txt")
)

if quesplit_files:
    quesplit_file = quesplit_files[0]
    with quesplit_file.open("r", encoding="utf-8") as f:
        unmasked_sents = [line.strip().split() for line in f if line.strip()]

    print(
        f"Unmasked sentences loaded from {quesplit_file.name}: "
        f"{len(unmasked_sents)}"
    )

    if len(q_sentences) != len(unmasked_sents):
        print(
            "WARNING: masked/unmasked sentence count mismatch "
            f"({len(q_sentences)} vs {len(unmasked_sents)}); "
            "unmasked visualization will be skipped."
        )
        unmasked_sents = None
else:
    print(
        "No De Machabeis quesplit file found for pattern: "
        "*machabeis_sentences.quesplit.txt"
    )

for candidate in candidates:
    print(f"\n{'=' * 60}")
    print(f"Processing candidate: {candidate}")
    print(f"{'=' * 60}")

    # -----------------------------------------------------
    # Candidate corpus
    # -----------------------------------------------------
    if candidate == "X4":
        known_corpus = certamina_corpus

        # X4 is itself a holdout. When it is the candidate, the
        # known-authorship test corpus remains the reference population.
        ref_corpus = test_corpus
        candidate_label = "X4_certamina"
    else:
        known_corpus = test_corpus.filter_by_level("author", candidate)

        # For ordinary background candidates, exclude the candidate
        # from the reference population.
        ref_corpus = TwoLevelCorpus(
            [s for s in test_corpus if s.author != candidate],
            levels=test_corpus.levels,
        )
        candidate_label = candidate

    known_total_tokens = sum(len(s.text.split()) for s in known_corpus)
    known_target_tokens = min(2000, known_total_tokens)

    if known_target_tokens == 0:
        print(f"Skipping {candidate}: no usable candidate text")
        continue

    known_sentences_raw, known_breakdown = known_corpus.sample_balanced(
        tokens=known_target_tokens,
        rng=rng,
    )
    known_sentences = [list(s.text.split()) for s in known_sentences_raw]

    print(
        f"Known sample: {sum(len(s) for s in known_sentences)} tokens "
        f"from {len(known_sentences)} sentences"
    )
    print(f"Known breakdown: {known_breakdown}")

    # -----------------------------------------------------
    # Reference corpus: 20,000 tokens
    # -----------------------------------------------------
    ref_total_tokens = sum(len(s.text.split()) for s in ref_corpus)
    ref_target_tokens = min(20000, ref_total_tokens)

    ref_sentences_raw, ref_breakdown = ref_corpus.sample_balanced(
        tokens=ref_target_tokens,
        rng=rng,
    )
    ref_sentences = [list(s.text.split()) for s in ref_sentences_raw]

    print(
        f"Reference sample: {sum(len(s) for s in ref_sentences)} tokens "
        f"from {len(ref_sentences)} sentences"
    )
    print(f"Reference breakdown: {ref_breakdown}")

    # -----------------------------------------------------
    # Masked visualization
    # -----------------------------------------------------
    masked_path = OUTPUT_DIR / f"lambdag_output_{candidate_label}_masked.html"

    result = lambdaG_visualize(
        q_text=q_sentences,
        k_text=known_sentences,
        ref_text=ref_sentences,
        N=8,
        r=30,
        output="html",
        scale="relative",
        negative=True,
        order_by="importance",
        rng=rng,
        print_path=str(masked_path),
    )

    print(f"Masked HTML saved to: {masked_path}")
    print(f"Table shape: {result['table'].shape}")
    print(f"Sentence count: {result['table']['sentence_id'].nunique()}")

    # -----------------------------------------------------
    # Unmasked visualization
    # -----------------------------------------------------
    if unmasked_sents is not None:
        unmasked_path = OUTPUT_DIR / f"lambdag_output_{candidate_label}_unmasked.html"

        result_unmasked = lambdaG_visualize(
            q_text=q_sentences,
            k_text=known_sentences,
            ref_text=ref_sentences,
            N=8,
            r=30,
            output="html",
            scale="relative",
            negative=True,
            order_by="",
            unmasked_sentences=unmasked_sents,
            rng=rng,
            print_path=str(unmasked_path),
        )

        print(f"Unmasked HTML saved to: {unmasked_path}")
    else:
        result_unmasked = None
        print("Skipping unmasked visualization")

    # -----------------------------------------------------
    # Summary statistics
    # -----------------------------------------------------
    table = result["table"]
    sentence_table = table.drop_duplicates("sentence_id")

    print("\nSummary statistics:")
    print(f"  Sentences: {table['sentence_id'].nunique()}")
    print(f"  Tokens: {len(table)}")
    print(f"  Mean sentence λG: {sentence_table['sentence_lambdaG'].mean():.3f}")
    print(f"  Std sentence λG: {sentence_table['sentence_lambdaG'].std():.3f}")
    print(
        f"  Positive tokens: {(table['lambdaG'] > 0).sum()} "
        f"({(table['lambdaG'] > 0).mean() * 100:.1f}%)"
    )
    print(
        f"  Negative tokens: {(table['lambdaG'] < 0).sum()} "
        f"({(table['lambdaG'] < 0).mean() * 100:.1f}%)"
    )


Found 38 background candidates + X4 (Certamina)
Questioned sentences (X3_machabeis): 213
Unmasked sentences loaded from X3_machabeis_sentences.quesplit.txt: 213

Processing candidate: AI
Known sample: 2013 tokens from 104 sentences
Known breakdown: {'AI': {'anticlaudianus': 2013}}
Reference sample: 20575 tokens from 1003 sentences
Reference breakdown: {'AeC': {'medicaminum': 7538, 'viaticus': 975}, 'AeP': {'karolinus': 469}, 'B': {'judas': 156}, 'BC': {'contemptu': 544}, 'DC': {'mathildis': 501}, 'E': {'biblia': 242}, 'FB': {'nuptis': 780}, 'GAp': {'roberti': 418}, 'GAr': {'philippis': 767}, 'GC': {'alexandreis': 775}, 'GM': {'merlini': 219}, 'GP': {'ligurinus': 868}, 'GVS': {'poetria': 308}, 'GiP': {'historia': 327}, 'H': {'epigrammata': 93, 'maria': 71}, 'HA': {'planctus': 294}, 'HP': {'maiolichinus': 438}, 'IH': {'architrenius': 510}, 'II': {'ylias': 403}, 'M': {'capitulorum': 297, 'maurilii': 31, 'victoris': 18}, 'NLC': {'passio': 242}, 'OM': {'ernestus': 363}, 'OSE': {'doctrina': 

## Stage 3: Optional combined diagnostic

The candidate-specific HTML files generated above are the outputs used for local interpretation. The following cell is retained only as an optional corpus diagnostic and does not enter the attribution results.


In [7]:
# OPTIONAL DIAGNOSTIC: inspect a balanced 2000-token sample
# from the complete known-authorship test corpus.

rng_combined = np.random.default_rng(seed=42)

test_total_tokens = sum(len(s.text.split()) for s in test_corpus)
sample_target_tokens = min(2000, test_total_tokens)

sample_raw, sample_breakdown = test_corpus.sample_balanced(
    tokens=sample_target_tokens,
    rng=rng_combined,
)

print(
    f"Optional combined sample: "
    f"{sum(len(s.text.split()) for s in sample_raw)} tokens "
    f"from {len(sample_raw)} sentences"
)
print(f"Breakdown: {sample_breakdown}")


Optional combined sample: 2669 tokens from 156 sentences
Breakdown: {'AI': {'anticlaudianus': 824}, 'AeC': {'medicaminum': 124, 'viaticus': 23}, 'AeP': {'karolinus': 57}, 'B': {'judas': 25}, 'BC': {'contemptu': 54}, 'DC': {'mathildis': 48}, 'E': {'biblia': 47}, 'FB': {'nuptis': 96}, 'GAp': {'roberti': 41}, 'GAr': {'philippis': 103}, 'GC': {'alexandreis': 83}, 'GM': {'merlini': 21}, 'GP': {'ligurinus': 91}, 'GVS': {'poetria': 27}, 'GiP': {'historia': 33}, 'H': {'epigrammata': 9, 'maria': 13}, 'HA': {'planctus': 35}, 'HP': {'maiolichinus': 47}, 'IH': {'architrenius': 75}, 'II': {'ylias': 41}, 'M': {'capitulorum': 31, 'maurilii': 31, 'victoris': 18}, 'NLC': {'passio': 62}, 'OM': {'ernestus': 62}, 'OSE': {'doctrina': 29}, 'RC': {'tancredi': 8}, 'RT': {'mauri': 38}, 'SG': {'ecclesiasten': 18}, 'U1': {'canticamariae': 5}, 'U10': {'karolellus': 16}, 'U11': {'katherinae': 38}, 'U2': {'liberesdre': 43}, 'U3': {'generatione': 82}, 'U4': {'dialogi': 56}, 'U5': {'britanniae': 60}, 'U6': {'roudlieb